In [0]:
%sql
CREATE OR REPLACE TABLE flightops_dev.gold.carrier_daily_performance
CLUSTER BY (flight_date, carrier)
COMMENT 'Daily on-time, delay, and completion metrics per carrier. On-time = operated flight arriving <= 15 min late.'
AS
SELECT
  flight_date,
  carrier,
  COUNT(*)                                                              AS scheduled_flights,
  SUM(CASE WHEN is_cancelled THEN 1 ELSE 0 END)                         AS cancelled_flights,
  SUM(CASE WHEN is_diverted  THEN 1 ELSE 0 END)                         AS diverted_flights,
  SUM(CASE WHEN NOT is_cancelled AND NOT is_diverted THEN 1 ELSE 0 END) AS operated_flights,
  SUM(CASE WHEN NOT is_cancelled AND NOT is_diverted
            AND arr_delay_min <= 15 THEN 1 ELSE 0 END)                  AS on_time_flights,
  ROUND(100 * try_divide(
      SUM(CASE WHEN NOT is_cancelled AND NOT is_diverted AND arr_delay_min <= 15 THEN 1 ELSE 0 END),
      SUM(CASE WHEN NOT is_cancelled AND NOT is_diverted THEN 1 ELSE 0 END)), 2) AS on_time_pct,
  ROUND(100 * try_divide(
      SUM(CASE WHEN NOT is_cancelled THEN 1 ELSE 0 END), COUNT(*)), 2)  AS completion_factor_pct,
  ROUND(AVG(CASE WHEN NOT is_cancelled AND NOT is_diverted THEN arr_delay_min END), 1) AS avg_arr_delay_min
FROM flightops_dev.silver.flights
GROUP BY flight_date, carrier;

In [0]:
%sql
CREATE OR REPLACE TABLE flightops_dev.gold.aa_hub_performance
COMMENT 'Monthly departure performance for American Airlines at its hubs.'
AS
SELECT
  origin                                    AS hub,
  DATE_TRUNC('MONTH', flight_date)          AS month,
  COUNT(*)                                  AS scheduled_departures,
  ROUND(100 * AVG(CASE WHEN NOT is_cancelled AND dep_delay_min <= 15 THEN 1.0
                       WHEN NOT is_cancelled THEN 0.0 END), 2)  AS dep_on_time_pct,
  ROUND(AVG(CASE WHEN NOT is_cancelled THEN dep_delay_min END), 1)  AS avg_dep_delay_min,
  ROUND(AVG(taxi_out_min), 1)               AS avg_taxi_out_min,
  ROUND(100 * AVG(CAST(is_cancelled AS INT)), 2) AS cancel_pct
FROM flightops_dev.silver.flights
WHERE carrier = 'AA'
  AND origin IN ('DFW','CLT','ORD','PHL','PHX','MIA','DCA','LAX','JFK','LGA')
GROUP BY origin, DATE_TRUNC('MONTH', flight_date);

In [0]:
%sql
CREATE OR REPLACE TABLE flightops_dev.gold.delay_cause_breakdown
COMMENT 'Monthly delay minutes by BTS cause per carrier (only flights with >=15 min arrival delay report causes).'
AS
SELECT
  carrier,
  DATE_TRUNC('MONTH', flight_date)         AS month,
  SUM(carrier_delay_min)                   AS carrier_min,
  SUM(weather_delay_min)                   AS weather_min,
  SUM(nas_delay_min)                       AS nas_min,
  SUM(security_delay_min)                  AS security_min,
  SUM(late_aircraft_delay_min)             AS late_aircraft_min,
  SUM(carrier_delay_min + weather_delay_min + nas_delay_min
      + security_delay_min + late_aircraft_delay_min) AS total_delay_min
FROM flightops_dev.silver.flights
WHERE NOT is_cancelled
GROUP BY carrier, DATE_TRUNC('MONTH', flight_date);

In [0]:
%sql
CREATE OR REPLACE TABLE flightops_dev.gold.route_performance
COMMENT 'Annual route-level performance per carrier; routes with at least one flight per day on average.'
AS
SELECT
  carrier, origin, dest,
  COUNT(*)                                                    AS scheduled_flights,
  ROUND(100 * AVG(CASE WHEN NOT is_cancelled AND NOT is_diverted AND arr_delay_min <= 15 THEN 1.0
                       WHEN NOT is_cancelled AND NOT is_diverted THEN 0.0 END), 2) AS on_time_pct,
  ROUND(AVG(CASE WHEN NOT is_cancelled AND NOT is_diverted THEN arr_delay_min END), 1) AS avg_arr_delay_min,
  ROUND(100 * AVG(CAST(is_cancelled AS INT)), 2)              AS cancel_pct,
  MAX(distance_miles)                                         AS distance_miles
FROM flightops_dev.silver.flights
GROUP BY carrier, origin, dest
HAVING COUNT(*) >= 365;

In [0]:
%sql
SELECT hub,
       SUM(scheduled_departures)                AS departures,
       ROUND(AVG(dep_on_time_pct), 1)           AS avg_dep_on_time_pct,
       ROUND(AVG(avg_dep_delay_min), 1)         AS avg_dep_delay_min,
       ROUND(AVG(cancel_pct), 2)                AS avg_cancel_pct
FROM flightops_dev.gold.aa_hub_performance
GROUP BY hub
ORDER BY avg_dep_on_time_pct;

In [0]:
%sql
SELECT carrier,
       ROUND(100 * SUM(carrier_min)       / SUM(total_delay_min), 1) AS carrier_pct,
       ROUND(100 * SUM(late_aircraft_min) / SUM(total_delay_min), 1) AS late_aircraft_pct,
       ROUND(100 * SUM(nas_min)           / SUM(total_delay_min), 1) AS nas_pct,
       ROUND(100 * SUM(weather_min)       / SUM(total_delay_min), 1) AS weather_pct,
       ROUND(100 * SUM(security_min)      / SUM(total_delay_min), 2) AS security_pct
FROM flightops_dev.gold.delay_cause_breakdown
WHERE carrier IN ('AA','DL','UA')
GROUP BY carrier;